# 01 – Monthly NDSI of Switzerland from Landsat (5, 7, 8, 9) and Sentinel-2 (SDC STAC catalogue)

Monthly NDSI statistics (`max`, `min`, `mean`, `count`, `qual`) at ~30 m from all clear Landsat and Sentinel-2 observations
of the Swiss Data Cube. This notebook keeps the structure of the original Open Data Cube notebook (1984–2021 products);
only the data access changes:

| Open Data Cube (original) | STAC (this notebook) |
|---|---|
| `dc = datacube.Datacube()` | `dc = open_catalog()` (SDC STAC catalogue) |
| `ls5_ledaps_swiss`, `ls7_ledaps_swiss`, `ls8_lasrc_swiss`, `s2_l2a_10m_swiss` | `landsat_tm_c2_l2` (L5, 1984–2011), `landsat_etm_c2_l2` (L7, only before the SLC failure of May 2003), `landsat_ot_c2_l2` (L8 + L9), `s2_l2` |
| Collection 1 `pixel_qa` (`ls_qa_clean`, categories 1, 2, 4) | Collection 2 `QA_PIXEL`, same rule: clear / water / snow with low cloud and cirrus confidence |
| Sentinel-2 `slc` (categories 4, 5, 6, 7, 11) | `SCL`, same categories |
| reflectance already scaled in the SDC | Landsat C2: DN × 0.0000275 − 0.2; Sentinel-2: DN / 10,000 |
| `utils.data_cube_utilities` / `swiss_utils` | `utils/sdc_stac_utilities.py` (same function names) |

Grid (the grid of the original products, 14234 × 6199 pixels of ~30 m, so that the time series has no shift), resampling
(cubic for the bands, nearest neighbour for the masks), 0.5 square-degree chunks, NDSI statistics and file names
`NDSI_<year>_<month>.tif` are **identical to the original**.

In [ ]:
import sys
sys.path.append("../utils")

In [ ]:
# Import dependencies

import os
import sys
import calendar
import subprocess

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from datetime import datetime, date
from time import time

# STAC versions of the former SDC utilities (same names)
from sdc_stac_utilities import open_catalog, get_platform, stac_load_product, geobox_from_grid, zone_coords
from sdc_stac_utilities import ls_qa_clean, create_slc_clean_mask
from sdc_stac_utilities import create_geographic_chunks, combine_geographic_chunks
from sdc_stac_utilities import printandlog, write_geotiff_from_xr, display_map

dc = open_catalog()          # replaces datacube.Datacube()

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Configuration section

# STAC collections of the SDC
# landsat_tm_c2_l2  = Landsat 5 (1984-2011)
# landsat_etm_c2_l2 = Landsat 7 (1999-2023)
# landsat_ot_c2_l2  = Landsat 8 and 9 (2013-)
# s2_l2             = Sentinel-2 (2015-)

products = ['landsat_tm_c2_l2', 'landsat_etm_c2_l2', 'landsat_ot_c2_l2', 's2_l2']
platforms = get_platform(products)

# period of use of each product (first day, last day)
# Landsat 7 only before the SLC failure of 31 May 2003 (striped images afterwards)
products_periods = {'landsat_tm_c2_l2':  ('1984-01-01', '2011-12-31'),
                    'landsat_etm_c2_l2': ('1999-01-01', '2003-05-30'),
                    'landsat_ot_c2_l2':  ('2013-01-01', '2100-12-31'),
                    's2_l2':             ('2015-06-01', '2100-12-31')}

# landsat_ot_c2_l2 contains Landsat 8 AND Landsat 9 (since 2022): both are used
# (put ['landsat-8'] to use only Landsat 8)
landsat_ot_platforms = ['landsat-8', 'landsat-9']

# buffer (in pixels of 30 m) around Sentinel-2 clouds / cloud shadows / cirrus (SCL 3, 8, 9, 10)
# 0 = no buffer, as in the datacube version
s2_cloud_buffer = 0

#products = ['landsat_ot_c2_l2']        # Landsat 8/9 only
#products = ['s2_l2']                   # Sentinel-2 only

measurements = ['green', 'swir1']       # mask band (pixel_qa or slc) is added by the load function

############### Switzerland all #############
min_lon =  5.7
max_lon =  10.55
min_lat = 45.74
max_lat = 47.85

############### small zone for test #############
#min_lon =  8
#max_lon =  8.15
#min_lat = 46.5
#max_lat = 46.65

start_year = 2022                  # 1984-2021 already produced with the datacube version
end_year = date.today().year

start_month = 1
end_month = 12

resolution = (0.0003407435000010011, 0.0003407435000010325)
resampling = {'*': 'cubic', 'pixel_qa': 'nearest', 'slc': 'nearest'}

# national pixel grid = grid of the former datacube products (e.g. NDSI_2020_11.tif, Landsat + Sentinel-2)
# -> the new months are exactly on the same pixels as the old series (no shift)
grid_origin = (5.699957268017272, 47.85026882419969)            # top-left corner (lon, lat)
grid_res    = (0.0003407435000010325, 0.00034046861818428666)   # pixel size (x, y) in degrees
grid_size   = (14234, 6199)                                     # (columns, rows)
gbox = geobox_from_grid(grid_origin, grid_res, grid_size)

work_path = '../outputs/NDSI_monthly_STAC'
log_name = '../outputs/NDSI_monthly_STAC.log'

In [ ]:
display_map(latitude = (min_lat, max_lat), longitude = (min_lon, max_lon))

## Switzerland mask

The Switzerland and lake masks are created once on the NDSI grid:
- **Switzerland:** from `data/sources/switzerland_border.shp` (swissBOUNDARIES3D, ICC = CH), 1 = Switzerland;
- **lakes:** from `data/sources/lakes.tif`, 1 = lake.

They have exactly the size of the grid (14234 × 6199), so no resizing is needed.

In [ ]:
mask_dir = '../data/aux'
os.makedirs(mask_dir, exist_ok=True)
h, w = gbox.shape
te = [str(v) for v in gbox.extent.boundingbox]          # left bottom right top

mask_ch_tif = '%s/mask_switzerland_%ix%i.tif' % (mask_dir, w, h)
if not os.path.exists(mask_ch_tif):
    subprocess.run(['ogr2ogr', '-overwrite', '-f', 'GPKG', '-t_srs', 'EPSG:4326', '-where', "ICC='CH'", '-dim', 'XY',
                    '%s/switzerland_wgs84.gpkg' % mask_dir, '../data/sources/switzerland_border.shp'], check=True)
    subprocess.run(['gdal_rasterize', '-burn', '1', '-init', '0', '-ot', 'Byte', '-te', *te, '-ts', str(w), str(h),
                    '-co', 'COMPRESS=DEFLATE', '%s/switzerland_wgs84.gpkg' % mask_dir, mask_ch_tif], check=True)

mask_lakes_tif = '%s/mask_lakes_%ix%i.tif' % (mask_dir, w, h)
if not os.path.exists(mask_lakes_tif):
    subprocess.run(['gdalwarp', '-overwrite', '-t_srs', 'EPSG:4326', '-te', *te, '-ts', str(w), str(h), '-r', 'near',
                    '-ot', 'Byte', '-srcnodata', 'None', '-dstnodata', 'None', '-co', 'COMPRESS=DEFLATE',
                    '../data/sources/lakes.tif', mask_lakes_tif], check=True)

In [ ]:
from osgeo import gdal

dataset = gdal.Open(mask_ch_tif)
print("Number of layers {}".format(dataset.RasterCount))
print("==== Some info ====")
print("Size of tif file is {} x {} x {}".format(dataset.RasterXSize,
                                    dataset.RasterYSize,
                                    dataset.RasterCount))
print("Projection is {}".format(dataset.GetProjection()))
Mask_CH = np.array(dataset.GetRasterBand(1).ReadAsArray())
print("The shape of the array is {} ".format(Mask_CH.shape))
# same grid as the NDSI -> coordinates attached, no resize needed
Mask_CH = xr.DataArray(Mask_CH, coords={'latitude': gbox.coords['latitude'].values, 'longitude': gbox.coords['longitude'].values}, dims=('latitude', 'longitude'))

In [ ]:
format(Mask_CH.shape)

In [ ]:
plt.imshow(Mask_CH)

## Lake mask
Note: lakes are 1, the pixels to keep are 0.

In [ ]:
dataset = gdal.Open(mask_lakes_tif)
print("Number of layers {}".format(dataset.RasterCount))
print("==== Some info ====")
print("Size of tif file is {} x {} x {}".format(dataset.RasterXSize,
                                    dataset.RasterYSize,
                                    dataset.RasterCount))
print("Projection is {}".format(dataset.GetProjection()))
Mask_lakes = np.array(dataset.GetRasterBand(1).ReadAsArray())
print("The shape of the array is {} ".format(Mask_lakes.shape))
Mask_lakes = xr.DataArray(Mask_lakes, coords={'latitude': gbox.coords['latitude'].values, 'longitude': gbox.coords['longitude'].values}, dims=('latitude', 'longitude'))

In [ ]:
format(Mask_lakes.shape)

In [ ]:
plt.imshow(Mask_lakes)

## Functions to load and mix Landsat and Sentinel-2

In [ ]:
def load_multi_clean_30(dc, products, time, lon, lat, measurements, resolution, resampling, dropna = True, platforms = [], valid_cats = []):
    ## load the data at 30 m resolution (STAC version)
    """
    Description:
      Create a clean dataset (multi-product or not) using cleaning "autor's recommended ways"
      - ls_qa_clean
      - create_slc_clean_mask
      Sorted by ascending time
      Works with Landsat or Sentinel 2 (but not mixed).
      dropna option removes time without any data
    Input:
      dc:           STAC catalogue (open_catalog())
    Args:
      products:     list of products (STAC collections)
      time:         pair (list) of minimum and maximum date
      lon:          pair (list) of minimum and maximum longitude
      lat:          pair (list) of minimum and maximum longitude
      measurements: list of measurements (must include pixel_qa or slc (not and !))
      resolution:   kept for compatibility, the grid is given by the national gbox
      resampling:   dict of resampling methods ('*' for the default)
      dropna:       if True removes times without any data
      valid_cats:   array of ints representing what category should be considered valid
                    * meand category by default
      # SENTINEL 2 ################################
      #   0 - no data                             #
      #   1 - saturated or defective              #
      #   2 - dark area pixels                    #
      #   3 - cloud_shadows                       #
      #   4 * vegetation                          #
      #   5 * not vegetated                       #
      #   6 * water                               #
      #   7 * unclassified                        #
      #   8 - cloud medium probability            #
      #   9 - cloud high probability              #
      #  10 - thin cirrus                         #
      #  11 * snow                                #
      #############################################
      # LANDSAT 5, 7, 8 and 9 #####################
      #    0 : Fill                               #
      #    1 * Clear                              #
      #    2 * Water                              #
      #    3 : Cloud shadow                       #
      #    4 * Snow                               #
      #    5 : Cloud                              #
      #############################################
    Output:
      cleaned dataset and clean_mask sorted by ascending time
    Authors:
      Bruno Chatenoux (UNEP/GRID-Geneva, 10.12.2019), adapted to STAC
    """

    # Convert product string into list
    if isinstance(products, str):
        products = products.split()
    # Check LANDSAT and SENTINEL products are not mixed
    prfx = list(set(['LANDSAT' if p.startswith('landsat') else 'SENTINEL' for p in products]))
    if len(prfx) > 1:
        sys.exit('Mixed platforms %s' % (set(prfx)))

    # Create raw dataset
    dataset_clean = None
    for product in products:
        dataset_tmp = stac_load_product(dc, product,
                                        time = time,
                                        lon = lon,
                                        lat = lat,
                                        measurements = measurements,
                                        gbox = gbox,
                                        resampling = resampling,
                                        period = products_periods.get(product),
                                        platforms_keep = landsat_ot_platforms if product == 'landsat_ot_c2_l2' else None)

        if len(dataset_tmp.variables) == 0: continue # skip the current iteration if empty

        # Clean dataset_tmp
        if prfx[0] == "LANDSAT":
            if len(valid_cats) == 0: valid_cats = [1, 2, 4]
            clean_mask_tmp = ls_qa_clean(dataset_tmp.pixel_qa, valid_cats)
        else:
            if len(valid_cats) == 0: valid_cats = [4, 5, 6, 7, 11]
            clean_mask_tmp = create_slc_clean_mask(dataset_tmp.slc, valid_cats, cloud_buffer = s2_cloud_buffer)
        dataset_clean_tmp = dataset_tmp.where(clean_mask_tmp)
        del dataset_tmp

        # Remove negative values
        dataset_clean_tmp = dataset_clean_tmp.where(dataset_clean_tmp >= 0)

        if dataset_clean is None:
            dataset_clean = dataset_clean_tmp.copy(deep=True)
        else:
            dataset_clean = xr.concat([dataset_clean, dataset_clean_tmp], dim = 'time')
        del dataset_clean_tmp

    #if dropna:
        # remove time without any data
        #dataset_clean = dataset_clean.dropna('time', how='all')

    if dataset_clean is not None:
        # Sort dataset by ascending time
        dataset_clean = dataset_clean.sortby('time')
        return (dataset_clean, ~np.isnan(dataset_clean[measurements[0]].values))
    else:
        return (0, 0)

In [ ]:
def load_lss2_clean_30(dc, products, time, lon, lat, measurements,
                    dropna = False, platforms = [], valid_cats = [[],[]]):
    ## load the Landsat and Sentinel-2 data at 30 m resolution in a dictionary (STAC version)
    """
    Description:
      Create a clean dataset mixing Landsat and Sentinel 2 products
      and using cleaning "autor's recommended ways":
      - ls_qa_clean
      - create_slc_clean_mask
      Sorted by ascending time
      Landsat and Sentinel 2 are loaded on the same national 30 m grid (gbox), so the Sentinel 2
      extent does not need to be adjusted on the Landsat one any more.
      A sensor without data for the period (e.g. Sentinel 2 before 2015) returns 0.
    Input:
      dc:           STAC catalogue (open_catalog())
    Args:
      products:     list of products (STAC collections)
      time:         pair (list) of minimum and maximum date
      lon:          pair (list) of minimum and maximum longitude
      lat:          pair (list) of minimum and maximum longitude
      measurements: list of measurements (without mask band)
      valid_cats:   list of list of ints representing what category should be considered valid
                    first Landsat categories, then Sentinel 2 categories
    Output:
      cleaned dataset and clean_mask sorted by ascending time stored in dictionnaries
      ('ls' and 's2')
    Authors:
      Bruno Chatenoux (UNEP/GRID-Geneva, 11.12.2019), adapted to STAC
    """

    # dictionnary sensor - mask band
    dict_sensmask = {'ls':'pixel_qa',
                     's2': 'slc'}

    assert (len(valid_cats) == 2), \
           '\nvalid_cats argument must be a list of list (read the doc for more details)'

    dict_dsc = {}
    dict_cm = {}

    # Process first Landsat and then Sentinel 2 (based on dict_sensmask order)
    for index, sensor in enumerate(dict_sensmask.keys()):
        sensor_products = [prod for prod in products if (prod.startswith('landsat') if sensor == 'ls' else prod.startswith('s2'))]
        if len(sensor_products) == 0:
            dict_dsc[sensor], dict_cm[sensor] = 0, 0
            continue
        # Remove mask bands if any
        measurements = [ele for ele in measurements if ele not in dict_sensmask.values()]
        # append proper mask band to measurements
        measurements.append(dict_sensmask[sensor])

        dsc, cm = load_multi_clean_30(dc = dc,
                                  products = sensor_products,
                                  time = time,
                                  lon = lon,
                                  lat = lat,
                                  measurements = measurements,
                                  resolution = resolution,
                                  resampling = resampling,
                                  dropna = dropna,
                                  valid_cats = valid_cats[index])
        dict_dsc[sensor] = dsc
        dict_cm[sensor] = cm

    return dict_dsc, dict_cm

### Cloud mask

Landsat Collection 2 (`QA_PIXEL`): `ls_qa_clean` (in `utils/sdc_stac_utilities.py`) applies the rule of the original Collection 1 version.
- **A pixel is kept if:**
  - the cloud confidence (and the cirrus confidence) is low;
  - there is no cloud, dilated cloud, cirrus, cloud shadow or fill;
  - the pixel is clear (1), water (2) or snow (4), with the Collection 1 numbering.
- **Collection 2 bits:** clear = bit 6, water = bit 7, snow = bit 5.

In [ ]:
#calculate NDSI
def NDSI(array):
    return (array.green - array.swir1)/(array.green + array.swir1)

## Monthly max, mean and min NDSI, Landsat and Sentinel-2 mixed

- **Resume:** a month already written in `work_path` is skipped, so the notebook can be run again without recomputing everything.
- **Current month:** it is not computed, only complete months are.
- **Months without any data:** nothing is written (as in the original).
- **Test:** to test, set the small zone and a single year / month in the configuration, and another `work_path`.

In [ ]:
def multi_ndsi(i):
    dict_dsc, dict_cm = load_lss2_clean_30(dc = dc,
                                    products = products ,
                                    time = [start_date, end_date],
                                    lon = geographic_chunks[i]['longitude'],
                                    lat = geographic_chunks[i]['latitude'],
                                    measurements = measurements,
                                    #dropna = True ,
                                    valid_cats = [[1,2,4],[4,5,6,7,11]])

    # sensors with data (Sentinel 2 only from 2015, Landsat 7 stops in 2023, ...)
    dsl = []
    if not isinstance(dict_dsc['ls'], int):
        dict_dsc['ls']['mask'] = dict_dsc['ls']['pixel_qa']
        dsl.append(dict_dsc['ls'].drop(['pixel_qa']))
    if not isinstance(dict_dsc['s2'], int):
        dict_dsc['s2']['mask'] = dict_dsc['s2']['slc']
        dsl.append(dict_dsc['s2'].drop(['slc']))

    if len(dsl) > 0:
        da = xr.concat(dsl, dim='time')
        ds = da.sortby('time')

        ndsi = (ds.green - ds.swir1) / (ds.green + ds.swir1)

        ndsi_count = ndsi.count(dim=['time'])
        ndsi_qual = np.rint((ndsi_count / len(ndsi['time']) * 100)).astype(np.uint8)
        del dict_dsc
        del dict_cm
        #max
        ndsi_max = ndsi.max(dim = ['time'])
        ndsi_max.name = 'ndsi_max'
        #min
        ndsi_min = ndsi.min(dim = ['time'])
        ndsi_min.name = 'ndsi_min'
        #mean
        ndsi_mean = ndsi.mean(dim = ['time'])
        ndsi_mean.name = 'ndsi_mean'

        # Append result
        ndsi_max_chunks.append(ndsi_max.to_dataset(name='max'))
        del ndsi_max
        ndsi_min_chunks.append(ndsi_min.to_dataset(name='min'))
        del ndsi_min
        ndsi_mean_chunks.append(ndsi_mean.to_dataset(name='mean'))
        del ndsi_mean

        ndsi_count_chunks.append(ndsi_count.to_dataset(name='count'))
        del ndsi_count
        ndsi_qual_chunks.append(ndsi_qual.to_dataset(name='qual'))
        del ndsi_qual


    else:
        print("N/A")

In [ ]:
%%time

# Create/Empty working directory
if not os.path.exists(work_path):
    os.makedirs(work_path)

# Cut the geographic extents into chunks
chunk_size = 0.5
geographic_chunks = create_geographic_chunks(latitude=(min_lat, max_lat),
                                             longitude=(min_lon, max_lon),
                                             geographic_chunk_size=chunk_size)
tot = len(geographic_chunks)

# coordinates of the processed zone on the national grid
zone_lat, zone_lon = zone_coords(gbox, (min_lon, max_lon), (min_lat, max_lat))

printandlog('NDSI max, min, mean by month calculation started', log_name)
start_time = datetime.now()
years = range(start_year, end_year + 1)
months = range(start_month, end_month + 1)
today = date.today()

for year in years:
    for month in months:
        # only complete months
        if (year, month) >= (today.year, today.month):
            continue
        # skip months already calculated
        out_tif = '%s/NDSI_%s_%02d.tif'% (work_path, year, month)
        if os.path.exists(out_tif):
            continue

        starty_time = datetime.now()
        print('Processing %i/%i' % (month, year))

        start_date = datetime.strptime("%i-%i-1" % (year, month), '%Y-%m-%d')
        # Need to find last days of a given month
        end_date = datetime.strptime("%i-%i-%i" % (year, month, calendar.monthrange(year, month)[1]), '%Y-%m-%d')

        ndsi_max_chunks = []
        ndsi_min_chunks = []
        ndsi_mean_chunks = []
        ndsi_count_chunks = []
        ndsi_qual_chunks = []

        bar_len = 20

        for i in range(tot):
            multi_ndsi(i)
            filled_len = round(bar_len * (i + 1) /  tot)
            # progress bar
            sys.stdout.write('\r[{0}{1}] in {2}'
                         .format('#' * filled_len, '-' * (bar_len - filled_len), datetime.now() - starty_time))
        sys.stdout.write('\r[{0}] in {1} - COMPLETED\n'.format('#' * bar_len, datetime.now() - starty_time))

        if len (ndsi_max_chunks) > 0:
            # Combine chunks
            with np.errstate(divide='ignore'):
                NDSI = combine_geographic_chunks(ndsi_max_chunks)
                NDSI = NDSI.merge(combine_geographic_chunks(ndsi_min_chunks))
                NDSI = NDSI.merge(combine_geographic_chunks(ndsi_mean_chunks))
                NDSI = NDSI.merge(combine_geographic_chunks(ndsi_count_chunks))
                NDSI = NDSI.merge(combine_geographic_chunks(ndsi_qual_chunks))

            del ndsi_max_chunks
            del ndsi_min_chunks
            del ndsi_mean_chunks
            del ndsi_count_chunks
            del ndsi_qual_chunks

            # same extent every month (chunks without data are empty)
            NDSI = NDSI.reindex(latitude=zone_lat, longitude=zone_lon)

            NDSI = NDSI.where((Mask_lakes.sel(latitude=NDSI.latitude, longitude=NDSI.longitude) ==0))
            NDSI = NDSI.where((Mask_CH.sel(latitude=NDSI.latitude, longitude=NDSI.longitude) ==1))

            write_geotiff_from_xr(out_tif, NDSI, list(NDSI.data_vars), compr='DEFLATE')
            printandlog('NDSI %s_%02d written' % (year, month), log_name, started = starty_time)


printandlog('NDSI monthly raw data calculated', log_name, started = start_time)


printandlog('ALL DONE\n', log_name)

In [ ]:
NDSI

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(24, 4))
for a, v in zip(ax, NDSI.data_vars):
    NDSI[v][::10, ::10].plot(ax=a, cmap='RdBu' if v in ['max', 'min', 'mean'] else 'viridis')
    a.set_title(v)
plt.tight_layout()